# ColBERTv2 H2H -- minimal HF, no pylate (T4 Colab)

Bypasses the entire pylate / sentence-transformers / torchcodec
dependency cascade that has cost us multiple Colab sessions.  This
notebook uses ONLY:

  - Colab's preinstalled torch + torchvision (matched, do not touch).
  - transformers (preinstalled).
  - safetensors, huggingface_hub (preinstalled).
  - vstash (cloned, editable install).

ColBERTv2 inference goes through `experiments/colbert_minimal.py`:
loads `colbert-ir/colbertv2.0` as a plain BertModel + a 768->128
linear projection (the architecture from Khattab & Zaharia 2022),
computes MaxSim via batched einsum.  ~80 LOC, no exotic deps.

**Do NOT** try to install pylate, sentence-transformers, or upgrade
torch in this notebook.  Each of those touches the matched cu128
stack and triggers a different ABI cascade (we have a memory
document logging five attempts that all failed differently).

Outputs land in `MyDrive/lme_h2h/`:
  - `lme_full_500_colbertv2.json` (~30-45 min on T4)
  - `beir_colbertv2.json` (~15-25 min on T4)

In [ ]:
# Cell 1: Setup -- clone the experiment branch.  Install NOTHING
# else.  Colab T4 ships torch + transformers + safetensors;
# anything we add risks rebreaking the matched CUDA stack.
BRANCH = "feature/longmemeval-retrain-experiments"

%cd /content
!rm -rf /content/vstash
!git clone --branch $BRANCH https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e . --no-deps
# Install only the vstash deps we exercise (sqlite-vec, fastembed,
# pydantic, typer, rich) without re-resolving torch / transformers.
!pip install -q --upgrade-strategy only-if-needed \
    'sqlite-vec' 'fastembed>=0.4' 'pydantic>=2' typer rich

import torch

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
print("torch:", torch.__version__)
print(
    "cuda:",
    torch.cuda.get_device_name(0),
    "| mem GB:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
)

# Sanity: colbert_minimal must import cleanly on the existing env.
import sys

sys.path.insert(0, "/content/vstash")
print("colbert_minimal import OK -- no pylate, no sentence-transformers needed.")

In [ ]:
# Cell 2: Drive mount up front so each retrieval cell can persist its
# JSON immediately, before any runtime hiccup.
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE_OUT = "/content/drive/MyDrive/lme_h2h"
os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs("/content/results", exist_ok=True)
print("Drive mounted, output dir:", DRIVE_OUT)

In [ ]:
# Cell 3: Download longmemeval_s for the chat-memory benchmark.
import os
from huggingface_hub import hf_hub_download

TARGET = "/content/vstash/experiments/data/longmemeval"
os.makedirs(TARGET, exist_ok=True)
p = hf_hub_download(
    "xiaowu0162/longmemeval", "longmemeval_s", repo_type="dataset", local_dir=TARGET
)
print(f"Downloaded ({os.path.getsize(p) / 1024 / 1024:.1f} MB) -> {p}")

## ColBERTv2 over LongMemEval-s (engine=minimal)

In [ ]:
# Cell 4: Per-question fresh in-memory index -> MaxSim -> session
# dedupe -> Recall@K.  ~30-45 min for 500 questions on T4.
import os
import time

os.chdir("/content/vstash")
t0 = time.perf_counter()
!python -m experiments.longmemeval_colbert \
    --all \
    --engine minimal \
    --device cuda \
    --encode-batch-size 32 \
    --output /content/results/lme_full_500_colbertv2.json
print(f"\n[ColBERT LME] wall: {time.perf_counter() - t0:.1f}s")
!cp /content/results/lme_full_500_colbertv2.json $DRIVE_OUT/
print("Saved to Drive.")

## ColBERTv2 over BEIR (5 datasets, engine=minimal)

In [ ]:
# Cell 5: Same metric definitions as experiments/beir_benchmark.py
# (NDCG@10, Recall@10, MRR), so the JSONs combine cleanly with the
# local vstash results via experiments/h2h_combine.py.
import os
import time

os.chdir("/content/vstash")
t0 = time.perf_counter()
!python -m experiments.beir_colbert \
    --datasets scifact nfcorpus fiqa scidocs arguana \
    --engine minimal \
    --device cuda \
    --output /content/results/beir_colbertv2.json
print(f"\n[ColBERT BEIR] wall: {time.perf_counter() - t0:.1f}s")
!cp /content/results/beir_colbertv2.json $DRIVE_OUT/
print("Saved to Drive.")

In [ ]:
# Cell 6: Sanity print of both outputs.
import json

lme = json.load(open("/content/results/lme_full_500_colbertv2.json"))["summary"]
print("ColBERTv2 (minimal HF) on LongMemEval-s (n=500, macro):")
for k in (1, 3, 5, 10, 20, 50):
    print(f"  R@{k:<3d} = {lme['macro'][f'recall@{k}']:.4f}")
print()
print("ColBERTv2 (minimal HF) on BEIR (NDCG@10):")
for r in json.load(open("/content/results/beir_colbertv2.json")):
    m = r["colbertv2"]
    print(
        f"  {r['dataset']:<10} NDCG@10={m['ndcg_10']:.4f}  Recall@10={m['recall_10']:.4f}  MRR={m['mrr']:.4f}"
    )
print("\nDownload these JSONs from Drive on the local Mac, then run:")
print("  python -m experiments.h2h_combine")